# Wireless Networking and Mobile Computing

### Prerequisites

This Jupyter Notebook has been tested with Visual Studio Code, running in a local Python environment.

**Installs**

In [ ]:
# --- Installs: ensure required packages are available ---
%pip install numpy pandas matplotlib --upgrade pip --quiet

**Imports**

In [ ]:
# --- Imports: libraries used in this notebook ---
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import subprocess
import re
import xml.etree.ElementTree as ET
import sys

**Declarations**

Set the variables below and run the generation cell. 

- `scenario` is the path to the `.xml` scenario file that will be simulated.
- `thrp_file` is the name of the `.csv` file for throughput.
- `offer_file` is the name of the `.csv` file for offer.
- `delay_file` is the name of the `.csv` file for delay.

In [ ]:
# --- Declarations: editable variables for students ---
scenario = ...
thrp_file = ...
offer_file = ...
delay_file = ...

### 1 Simulation Results

#### 1.1. Folder Location
We dynamically extract the folder location from the property `Path2Results`.

In [ ]:
# --- Locate Results Folder ---
tree = ET.parse(scenario)
root = tree.getroot()
path_elem = root.find(".//JE802StatEval")
path2results = path_elem.get("Path2Results") if path_elem is not None else None

if not path2results:
    raise ValueError("Path2Results not found in scenario XML")

print(f"Found results folder: {path2results}")

# List contents (cross-platform)
if os.path.isdir(path2results):
    print(f"\nContents of {path2results}:")
    for f in os.listdir(path2results)[:10]:
        print("  - ", f)
    print("  -  ...")
else:
    print(f"\nResults folder {path2results} does not exist yet.")

# Change Directory to Results Folder
# Path2Results is already the folder the simulator wrote to, relative to where
# it ran, which is this notebook's directory. Use it as it stands: it is
# "Results/<name>" now, so taking its basename would point at nothing.
results_dir = path2results
if os.path.isdir(results_dir):
    os.chdir(results_dir)
    print(f"Changed directory to {results_dir}")
else:
    print(f"Results folder {results_dir} not found.")

#### 1.2 Google Earth Visualization
Locate it, and find the kml file to illustrate the scenario with Google Earth Pro ([Desktop version, not the Web version](https://www.google.com/earth/about/versions/#earth-pro)). This free version is available on Microsoft Windows, macOS, and GNU/Linux. When opening the kml file with Google Earth Pro, a scenario with two radio devices like the one in the attached image below should show up.

<p align="center" width="100%"><img width=80%" height="auto" src="../Admin/GettingStartedGoogleEarth.png">

In [ ]:
# --- Open KML File ---
kml_files = [f for f in os.listdir() if f.endswith(".kml")]
if kml_files:
    kml_path = os.path.abspath(kml_files[0])
    print(f"Opening {kml_path} ...")
    if sys.platform.startswith("win"):
        os.startfile(kml_path)
    elif sys.platform == "darwin":
        subprocess.run(["open", kml_path])
    else:
        subprocess.run(["xdg-open", kml_path])
else:
    print("No .kml file found in results folder.")


### 2 Statistical Evaluation

#### 2.1 Plotting Functions

In [ ]:
# --- Plotting & Helper Functions ---
def safe_read_csv(path, **kw):
    try: return pd.read_csv(path, **kw) if isinstance(path, (str, bytes, os.PathLike)) and path not in (None, ..., "") and os.path.exists(path) else None
    except: return None


def plot_thrp_offer_delay(result_total_offer, result_total_thrp, result_total_delay):
    fig, ax = plt.subplots(figsize=(8, 4))
    theLineWidth = 2
    ax.cla()
    ax.grid(color="#AAAAAA", linewidth=.5, linestyle='-', alpha=1)
    ax.axis([None, None, None, None])
    ax.set_facecolor((.2, 0.2, 0.3))
    plt.xlim(0, np.max(result_total_offer.iloc[:, 0]) / 1000 * 1.05)  # Set x-axis limits
    plt.xlabel('Simulated Time [s]')
    plt.ylabel('Simulation Results, Mb/s & ms')
    plt.grid(True)  # Turn on the grid
    plt.minorticks_on()  # Enable minor ticks

    max_time = 0
    if result_total_offer is not None:
        plt.plot(result_total_offer.iloc[:, 0] / 1000, result_total_offer.iloc[:, 6],
                 '-', color='g', linewidth=4, label='Offer [Mb/s]')
        max_time = max(max_time, np.max(result_total_offer.iloc[:, 0]))
    if result_total_thrp is not None:
        plt.plot(result_total_thrp.iloc[:, 0] / 1000, result_total_thrp.iloc[:, 6],
                 '-', color='w', linewidth=1, label='Throughput [Mb/s]')
        max_time = max(max_time, np.max(result_total_thrp.iloc[:, 0]))
    if result_total_delay is not None:
        plt.plot(result_total_delay.iloc[:, 0] / 1000, result_total_delay.iloc[:, 3],
                 '-', color='r', linewidth=1, label='Averg. Delay [ms]')
        max_time = max(max_time, np.max(result_total_delay.iloc[:, 0]))

    if max_time > 0:
        plt.xlim(0, max_time / 1000 * 1.05)

    plt.legend(fancybox=True, framealpha=1, facecolor=(.6, 0.61, 0.8))
    plt.tight_layout()

def plot_delay_histogram(delay):
    if delay is None or delay is ...:
        return

    try:
        aBinSize_ms_string, aHistogram_string = delay.iloc[-1, 7], delay.iloc[-1, 8]
    except Exception:
        return

    if not aBinSize_ms_string or not aHistogram_string or aBinSize_ms_string in (...,) or aHistogram_string in (...,):
        return

    fig, ax_distr = plt.subplots(figsize=(8, 4))
    ax_distr.cla()
    ax_distr.grid(color="#AAAAAA", linewidth=.5, linestyle='-', alpha=1)
    ax_distr.axis([None, None, None, None])
    ax_distr.set_facecolor((.2, 0.2, 0.3))

    theHistogram = [float(x) for x in aHistogram_string.split()]
    if not theHistogram or sum(theHistogram) == 0:
        return

    theHistogram = [x / sum(theHistogram) for x in theHistogram]
    theBinSize_ms = float(aBinSize_ms_string)
    theNumOfBins = len(theHistogram)
    theX = [i * theBinSize_ms + theBinSize_ms / 2 for i in range(theNumOfBins)]
    theCumSum = 1 - np.cumsum(theHistogram)
    theCumSum = np.insert(theCumSum, 0, 1)

    plt.step([0] + theX, [0] + theHistogram, '-', color='r', linewidth=2, label='PDF')
    plt.step([0] + theX, theCumSum, '-', color='m', linewidth=2, label='CDF')
    plt.grid(True)
    plt.minorticks_on()
    plt.xlim(0, theNumOfBins * theBinSize_ms * 1.05)
    plt.xlabel('x [ms]')
    plt.ylabel('PDF: P(delay=x) & CCDF: P(delay≥x)')
    plt.legend(fancybox=True, framealpha=1, facecolor=(.6, 0.61, 0.8))
    plt.tight_layout()


#### 2.2 Total System Plots

In [ ]:
# --- Plot Total Results ---
offer = safe_read_csv(offer_file, skiprows=2)
thrp = safe_read_csv(thrp_file, skiprows=2)
delay = safe_read_csv(delay_file, skiprows=2)

plot_thrp_offer_delay(offer, thrp, delay)
plot_delay_histogram(delay)